# Exhibition interpolation — gh0st_flux_lora_v2

Assembles Screen A and Screen B exhibition videos from the FLUX-generated stills
produced by `generate_exhibition.py`, with category text overlay from `text_payload.json`.

**Screen A:** stable → ambiguous → glitch → extreme → synthetic → `_node_a` text  
**Screen B:** extreme → synthetic → stable → ambiguous → glitch → `_node_b` text

**Sequence structure per category (N styles):** 3N−2 stills  
pure → 030mashup → 070mashup → pure → … → pure  
plus 2 boundary mashups between each category pair.

**Timing:** each still held for `HOLD_FRAMES`, then `MORPH_FRAMES` of optical flow to next.  
Default: 48+48 frames @ 24fps = 2s hold + 2s morph ≈ **10 min total**.

**Before running:** upload the generated stills to Drive:
```
Local:  spikes/flux_lora_training/output/gh0st_exhibition_v1/
Drive:  Gh0st in the Loop/outputs/gh0st_exhibition_v1/
```

In [ ]:
import os

!pip install opencv-python-headless imageio imageio-ffmpeg -q

from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(repo_dir := '/content/gh0st-in-the-l00p'):
    !git -C {repo_dir} pull --quiet
else:
    !git clone --quiet https://github.com/jasonr2048/gh0st-in-the-l00p.git {repo_dir}

%cd {repo_dir}
print('Ready.')

In [ ]:
from datetime import datetime
from pathlib import Path
import re, json

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive/Gh0st in the Loop')
STILLS_BASE   = DRIVE_ROOT / 'outputs' / 'gh0st_exhibition_v1'
SELECTION     = Path('spikes/flux_lora_training/exhibition_source/selection_v1.txt')
TEXT_PAYLOAD  = DRIVE_ROOT / 'Text' / 'text_payload.json'
OUTPUT_DIR    = DRIVE_ROOT / 'outputs'

# ── Screen sequences ──────────────────────────────────────────────────────────
SCREEN_A = ['stable', 'ambiguous', 'glitch', 'extreme', 'synthetic']
SCREEN_B = ['extreme', 'synthetic', 'stable', 'ambiguous', 'glitch']

# ── Timing ────────────────────────────────────────────────────────────────────
FPS          = 24
HOLD_FRAMES  = 48    # frames each still is held static  (48 = 2s)
MORPH_FRAMES = 48    # optical flow frames to next still  (48 = 2s)
# Total per still = HOLD_FRAMES + MORPH_FRAMES = 4s
# 154 stills × 4s ≈ 10 min (last still has no morph, but close enough)

# ── Text overlay ──────────────────────────────────────────────────────────────
FONT_SIZE    = 20    # px, adjust to taste
FONT_PATH    = 'fonts/CourierPrime-Regular.ttf'   # bundled in repo

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_SIZE  = (1024, 1024)   # (width, height)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_A = OUTPUT_DIR / f'exhibition_screen_A_{timestamp}.mp4'
OUT_B = OUTPUT_DIR / f'exhibition_screen_B_{timestamp}.mp4'

print(f'Screen A → {OUT_A.name}')
print(f'Screen B → {OUT_B.name}')
print(f'Hold: {HOLD_FRAMES/FPS:.1f}s  Morph: {MORPH_FRAMES/FPS:.1f}s  Per still: {(HOLD_FRAMES+MORPH_FRAMES)/FPS:.1f}s')

In [ ]:
# ── Parse selection file + text payload ───────────────────────────────────────

def stem(name: str) -> str:
    s = Path(name).stem
    return re.sub(r'[\s()]+', '_', s).strip('_')

def parse_selection(path: Path) -> dict:
    categories = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split('/', 1)
        if len(parts) != 2:
            continue
        cat, fname = parts
        categories.setdefault(cat, []).append(fname)
    return categories

selection   = parse_selection(SELECTION)
text_payload = json.loads(TEXT_PAYLOAD.read_text())

for cat, files in selection.items():
    n_stills = 3 * len(files) - 2
    print(f'  {cat}: {len(files)} styles → {n_stills} stills')

print(f'\nText keys available: {[k for k in text_payload if "node" in k]}')

In [ ]:
# ── Build ordered still lists + per-frame text schedule ───────────────────────

def build_sequence(screen_order: list, selection: dict, base: Path, node_suffix: str):
    """
    Returns:
      stills      : list of Paths
      text_events : list of (frame_index, message_str)  — one per category
                    frame_index is the first frame of that category's section
    """
    stills = []
    text_events = []   # (frame_start, [messages])

    for i, cat in enumerate(screen_order):
        files  = selection[cat]
        stems  = [stem(f) for f in files]
        cat_dir = base / cat
        cat_start_still = len(stills)   # index of first still in this category

        # First pure still
        stills.append(cat_dir / f'{stems[0]}.png')

        # Within-category: mashup pair + next pure
        for j in range(len(stems) - 1):
            sa, sb = stems[j], stems[j + 1]
            stills.append(cat_dir / f'030{sa}__070{sb}.png')
            stills.append(cat_dir / f'070{sa}__030{sb}.png')
            stills.append(cat_dir / f'{sb}.png')

        # Frame index where this category starts
        frame_start = cat_start_still * (HOLD_FRAMES + MORPH_FRAMES)
        messages = text_payload.get(f'{cat}_{node_suffix}', [])
        text_events.append((frame_start, cat, messages))

        # Boundary to next category
        if i < len(screen_order) - 1:
            next_cat = screen_order[i + 1]
            last_a  = stems[-1]
            first_b = stem(selection[next_cat][0])
            bdir = base / 'boundaries' / f'{cat}_x_{next_cat}'
            stills.append(bdir / f'030{last_a}__070{first_b}.png')
            stills.append(bdir / f'070{last_a}__030{first_b}.png')

    return stills, text_events


seq_a, text_a = build_sequence(SCREEN_A, selection, STILLS_BASE, 'node_a')
seq_b, text_b = build_sequence(SCREEN_B, selection, STILLS_BASE, 'node_b')

def report(name, seq, text_events):
    missing = [p for p in seq if not p.exists()]
    n = len(seq)
    total_frames = n * HOLD_FRAMES + (n - 1) * MORPH_FRAMES
    dur = total_frames / FPS
    print(f'{name}: {n} stills, {len(missing)} missing → {dur:.0f}s ({dur/60:.1f} min)')
    for frame_start, cat, msgs in text_events:
        print(f'  {frame_start/FPS:6.1f}s  {cat}  ({len(msgs)} messages)')
    if missing:
        print(f'  MISSING:')
        for m in missing:
            print(f'    {m.relative_to(STILLS_BASE)}')

report('Screen A', seq_a, text_a)
print()
report('Screen B', seq_b, text_b)

In [ ]:
# ── Preview: every 12th still from each screen ────────────────────────────────
from PIL import Image
import IPython.display as ipd

def thumb_strip(seq, step=12, label=''):
    sample = [p for p in seq[::step] if p.exists()]
    if not sample:
        print(f'{label}: no stills found')
        return
    thumbs = [Image.open(p).convert('RGB').resize((128, 128)) for p in sample]
    grid = Image.new('RGB', (len(thumbs) * 128, 128))
    for i, t in enumerate(thumbs):
        grid.paste(t, (i * 128, 0))
    print(label)
    ipd.display(grid)

thumb_strip(seq_a, label='Screen A (every 12th)')
thumb_strip(seq_b, label='Screen B (every 12th)')

In [ ]:
# ── Optical flow + text overlay helpers ───────────────────────────────────────
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

def load_img(path: Path, size: tuple) -> np.ndarray:
    img = Image.open(path).convert('RGB')
    w, h = size
    src_w, src_h = img.size
    if abs(src_w / src_h - w / h) > 0.01:
        if src_w / src_h > w / h:
            nw = int(src_h * w / h)
            img = img.crop(((src_w - nw) // 2, 0, (src_w - nw) // 2 + nw, src_h))
        else:
            nh = int(src_w * h / w)
            img = img.crop((0, (src_h - nh) // 2, src_w, (src_h - nh) // 2 + nh))
    return np.array(img.resize((w, h), Image.LANCZOS))


def optical_flow_morph(a: np.ndarray, b: np.ndarray, steps: int) -> list:
    ag = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    bg = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY)
    flow = cv2.calcOpticalFlowFarneback(
        ag, bg, None, pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
    h, w = a.shape[:2]
    xs = np.tile(np.arange(w), (h, 1)).astype(np.float32)
    ys = np.tile(np.arange(h), (w, 1)).T.astype(np.float32)
    frames = []
    for t in np.linspace(0, 1, steps, endpoint=False):
        warped = cv2.remap(a,
            (xs + flow[..., 0] * t).astype(np.float32),
            (ys + flow[..., 1] * t).astype(np.float32),
            cv2.INTER_LINEAR)
        frames.append(cv2.addWeighted(warped, 1 - t, b, t, 0))
    return frames


def _get_font(size):
    try:
        return ImageFont.truetype(FONT_PATH, size)
    except Exception:
        return ImageFont.load_default()


def build_text_schedule(text_events: list, total_frames: int) -> list:
    """
    Returns list of length total_frames, each entry is the text string to
    display at that frame (or None if no text). Messages within a category
    are spread evenly across that category's frame range, each shown for
    long enough to be readable before the next one appears.
    """
    schedule = [None] * total_frames

    for idx, (frame_start, cat, messages) in enumerate(text_events):
        if not messages:
            continue
        # Frame range for this category
        if idx + 1 < len(text_events):
            frame_end = text_events[idx + 1][0]
        else:
            frame_end = total_frames
        cat_frames = frame_end - frame_start
        interval = cat_frames // len(messages)   # frames per message
        for j, msg in enumerate(messages):
            f = frame_start + j * interval
            if f < total_frames:
                schedule[f] = msg

    return schedule


def draw_text(frame_rgb: np.ndarray, text: str, font, current_text: list) -> np.ndarray:
    """
    Overlay text on a frame (bottom-left, green terminal style).
    current_text is a mutable list holding the currently displayed string
    for the typewriter effect across frames.
    """
    if text is not None:
        current_text[0] = text
    if not current_text[0]:
        return frame_rgb

    img = Image.fromarray(frame_rgb)
    draw = ImageDraw.Draw(img)
    w, h = img.size
    msg = current_text[0]
    # Shadow for readability
    draw.text((22, h - 58), msg, font=font, fill=(0, 0, 0))
    draw.text((20, h - 60), msg, font=font, fill=(0, 255, 70))
    return np.array(img)

print('Helpers defined.')

In [ ]:
# ── Render function ───────────────────────────────────────────────────────────
import imageio

def render(seq: list, text_events: list, out_path: Path, label: str) -> float:
    existing = [p for p in seq if p.exists()]
    if not existing:
        raise RuntimeError(f'No stills found at {STILLS_BASE}')
    missing = len(seq) - len(existing)
    if missing:
        print(f'WARNING: {missing} stills missing — they will be skipped')

    n = len(existing)
    total_frames = n * HOLD_FRAMES + (n - 1) * MORPH_FRAMES
    print(f'Loading {n} stills...')
    imgs = [load_img(p, OUTPUT_SIZE) for p in existing]
    print('Loaded. Building text schedule...')

    text_schedule = build_text_schedule(text_events, total_frames)
    font = _get_font(FONT_SIZE)
    current_text = [None]   # mutable so draw_text can update it across frames

    print(f'Rendering {total_frames} frames ({total_frames/FPS:.1f}s) → {out_path.name}')
    out_path.parent.mkdir(parents=True, exist_ok=True)

    frame_idx = 0
    with imageio.get_writer(str(out_path), fps=FPS) as writer:
        for i, img in enumerate(imgs):
            # Hold frames
            for _ in range(HOLD_FRAMES):
                f = draw_text(img, text_schedule[frame_idx], font, current_text)
                writer.append_data(f)
                frame_idx += 1
            # Morph frames to next still
            if i < len(imgs) - 1:
                for morph_frame in optical_flow_morph(img, imgs[i + 1], MORPH_FRAMES):
                    f = draw_text(morph_frame, text_schedule[frame_idx], font, current_text)
                    writer.append_data(f)
                    frame_idx += 1
            if i % 20 == 0:
                print(f'  still {i}/{n}, frame {frame_idx}/{total_frames}', end='\r')

    dur = total_frames / FPS
    print(f'\n✅ {label} → {out_path.name}  ({dur:.0f}s / {dur/60:.1f} min)')
    return dur

print('render() defined.')

In [ ]:
# ── Render Screen A ───────────────────────────────────────────────────────────
dur_a = render(seq_a, text_a, OUT_A, 'Screen A')

In [ ]:
# ── Render Screen B ───────────────────────────────────────────────────────────
dur_b = render(seq_b, text_b, OUT_B, 'Screen B')

In [ ]:
# ── Sidecar JSON ──────────────────────────────────────────────────────────────
for out_path, dur, screen, seq, text_ev in [
    (OUT_A, dur_a, 'A', seq_a, text_a),
    (OUT_B, dur_b, 'B', seq_b, text_b),
]:
    sidecar = {
        'experiment_id': out_path.stem,
        'screen': screen,
        'duration_seconds': round(dur, 3),
        'fps': FPS,
        'source': 'gh0st_exhibition_v1_flux_lora_v2',
        'n_stills': len(seq),
        'hold_frames': HOLD_FRAMES,
        'morph_frames': MORPH_FRAMES,
        'output_size': OUTPUT_SIZE,
        'category_timings': [
            {'category': cat, 'start_seconds': round(f / FPS, 2), 'n_messages': len(msgs)}
            for f, cat, msgs in text_ev
        ],
        'generated_at': datetime.now().isoformat(),
    }
    sp = out_path.with_suffix('.json')
    sp.write_text(json.dumps(sidecar, indent=2))
    print(f'✅ {sp.name}')
    print(json.dumps(sidecar, indent=2))
    print()

In [ ]:
# ── In-notebook preview (Screen A) ───────────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode

data_url = 'data:video/mp4;base64,' + b64encode(open(OUT_A, 'rb').read()).decode()
display(HTML(f'<video width=540 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))